# Metano TROPOMI × infraestrutura de O&G no Nordeste — pipeline v4 (Google Colab)

Executa exatamente o mesmo código do repositório (`ch4ne/`). Etapas:
1. preparar o ambiente;
2. autenticar no Earth Engine;
3. fazer uma checagem rápida;
4. exportar em duas fases (background mensal, depois estatísticas);
5. aguardar;
6. baixar, com registro de proveniência;
7. rodar a análise local.

**Política de dados:** somente dados reais. A análise se recusa a rodar se os rasters não vierem da exportação do Earth Engine (`PROVENANCE.json` com checksums).

**Se o Colab desconectar:** rode as células 1 e 2 e continue da etapa em que parou. Cada célula de etapa é independente, e as tarefas do GEE seguem rodando no servidor.

## 1. Ambiente e parâmetros da rodada
Use **uma** das opções: (a) a pasta `pipeline_v4` no seu Google Drive; (b) o repositório no GitHub (troque a URL). Nos dois casos, as saídas ficam no Drive.

In [ ]:
import os, sys, json
USE_DRIVE_REPO = True
DRIVE_REPO = '/content/drive/MyDrive/pipeline_v4'             # pasta do repositório no Drive
GITHUB_URL = 'https://github.com/<usuario>/<repositorio>.git'   # se USE_DRIVE_REPO = False
OUTPUTS = '/content/drive/MyDrive/ch4ne_outputs/outputs_v4'     # saídas sempre no Drive

GEE_PROJECT = 'deep-presence-470415-u2'   # projeto do Google Cloud com Earth Engine habilitado
RUN_LABEL, START, END = '', '2020-01-01', '2025-12-31'   # robustez: ('P2', '2022-07-17', '2025-12-31')

from google.colab import drive
drive.mount('/content/drive')
if USE_DRIVE_REPO:
    REPO = DRIVE_REPO
else:
    REPO = '/content/repo'
    if not os.path.exists(REPO):
        !git clone -q $GITHUB_URL $REPO
os.chdir(REPO)
sys.path.insert(0, REPO)
!pip install -q -r requirements.txt

from ch4ne.config import Config
cfg = Config(base_dir=REPO, out_dir_name=OUTPUTS, run_label=RUN_LABEL, start_date=START, end_date=END,
             gee_project=GEE_PROJECT).validate()
print('Repositório:', REPO, '| saídas:', cfg.out_dir)

## 2. Earth Engine (login da sua conta Google)

In [ ]:
import ee
from ch4ne import gee_export as G
from ch4ne.boundaries import load_states, ne_states
try:
    ee.Initialize(project=cfg.gee_project)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=cfg.gee_project)
ne = ne_states(load_states(cfg), cfg)
print('GEE OK:', ee.String('conectado').getInfo())

## 3. Checagem rápida (cerca de 1 minuto)
Mostra as bandas, a contagem de órbitas, as versões do processador, uma órbita preparada e o background de um mês.

In [ ]:
G.dry_run(cfg, ne)

## 4. Exportar (fase A → espera → fase B)
A fase A grava os backgrounds mensais. A fase B grava `stats` (estatísticas por célula, numa passagem só), `monthly_AAAA` (um asset por ano) e `legacy`. As covariáveis estáticas são calculadas no download, com `computePixels`.

A exportação pode ser retomada: assets prontos com a mesma identidade são reaproveitados, e tarefas em execução são aguardadas. Acompanhe em https://code.earthengine.google.com/tasks. Para recalcular, use `OVERWRITE = True`; para exportar só alguns assets, use por exemplo `JOBS = ['stats']`.

In [ ]:
OVERWRITE = False
JOBS = None   # None = todos
ta = G.start_background_export(cfg, ne, overwrite=OVERWRITE)
G.wait(cfg, ta, optional=())
tb = G.start_main_exports(cfg, ne, overwrite=OVERWRITE, only=JOBS)
cfg.out_dir.mkdir(parents=True, exist_ok=True)
(cfg.out_dir / 'run_config.json').write_text(json.dumps(dict(gee_params=cfg.gee_params(),
    gee_project=cfg.gee_project, config=cfg.as_dict(), tasks_phase_a=ta, tasks_phase_b=tb), indent=2))

## 5. Aguardar a fase B (pode ser retomada depois de uma desconexão)

In [ ]:
m = json.loads((cfg.out_dir / 'run_config.json').read_text())
falhas_opcionais = G.wait(cfg, m['tasks_phase_b'])   # falha em stats/covariates interrompe; em monthly_* só avisa
falhas_opcionais

## 6. Baixar (GeoTIFF + CSV + PROVENANCE.json)

In [ ]:
files = G.download_all(cfg, cfg.out_dir)
{k: str(v) for k, v in files.items()}

## 7. Análise local (tabelas, figuras, números e checagens)

In [ ]:
from ch4ne.analysis import run
res = run(cfg, cfg.out_dir / 'gee', cfg.out_dir, sensitivity=True)
print(json.dumps(res['checks'], indent=2, default=str))

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(cfg.out_dir / 'tables' / 'Table3_clusters.csv').head(20))
display(pd.read_csv(cfg.out_dir / 'tables' / 'Table4_attribution_enrichment.csv'))
for f in ['Fig02_composite_and_data_density', 'Fig04_anomaly_and_clusters', 'Fig07_proximity_rings']:
    display(Image(filename=str(cfg.out_dir / 'figures' / f'{f}.png'), width=900))